# Standalone DIMSIM geometry analysis

All canonical implementation code is embedded as ordinary cells. No project checkout is required.


In [ ]:
BUILD_METADATA = {'schema_version': 1, 'baseline_git_sha': '1f77df1526f79a9b67fd092adad79dbbf24b8d79', 'source_sha256': {'evaluation/dimsim.py': '38712d92358e8378539935fd669f5e9a375193010007a3798faaea02c88e0195', 'evaluation/sampling.py': '39c473e643f1f30c62951d28d4e5b6004ed819256676ff351355d098ebeab0c8', 'evaluation/bundle.py': '0070c4f280ceb0e5802577fa58faf25e107f6e087790b5ba29974f1c0874d41d', 'evaluation/representations.py': '8d41e607938a592559f77a4e3900ddda3c15975017d0589fbad59a634c43ddbb', 'model.py': '9f6bd2de95cec891c875280e518726ec1c9f6f0ec6080b55fe564a88170415af', 'evaluation/encoders.py': '93444a8a740948ca06632cf68792377995fd0d88f6b152f68640e95d21a04aa4', 'evaluation/analysis.py': 'c06233981812e189e8b696c47fbe5d53d665a3d20bb611f0db15dffc12345577', 'evaluation/runner.py': 'a48b4e0478cf021a89f460641e94c559ca5f50f223e3dc7596a3a47e53d81e35'}, 'embedded_sha256': '80c663e38d9337b9b974075b9c03210df928051a23fb7b02411ba6ce6e7f4b47', 'generator_sha256': '496db360a59be5f2a418dcf1e37b209409c49dde9154885a1c2a8d5f314b1867'}


In [ ]:
import importlib.metadata
import subprocess
import sys

INSTALL_DEPENDENCIES = False
DEPENDENCIES = ["dimsim==0.2.2", "pypinyin==0.55.0", "numpy", "scipy", "pandas", "matplotlib"]
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *DEPENDENCIES])

for package in ("dimsim", "pypinyin", "numpy", "scipy", "pandas", "matplotlib"):
    print(package, importlib.metadata.version(package))
if importlib.metadata.version("dimsim") != "0.2.2" or importlib.metadata.version("pypinyin") != "0.55.0":
    raise RuntimeError("Pinned DIMSIM preparation dependencies do not match")


## Embedded `evaluation/dimsim.py`


In [ ]:
"""Pinned public DIMSIM adapter used by evaluation-bundle preparation."""
import hashlib
from importlib import metadata
from pathlib import Path
DIMSIM_VERSION = '0.2.2'
PYPINYIN_VERSION = '0.55.0'

class DimsimDistance:
    """Validated callable around the public ``dimsim.get_distance`` API."""

    def __init__(self) -> None:
        versions = {'dimsim': metadata.version('dimsim'), 'pypinyin': metadata.version('pypinyin')}
        expected = {'dimsim': DIMSIM_VERSION, 'pypinyin': PYPINYIN_VERSION}
        if versions != expected:
            raise RuntimeError(f'DIMSIM dependency versions {versions} do not match pinned {expected}')
        import dimsim
        if not callable(getattr(dimsim, 'get_distance', None)):
            raise RuntimeError('dimsim.get_distance public API is unavailable')
        self._get_distance = dimsim.get_distance
        package_root = Path(dimsim.__file__).resolve().parent
        source_hashes = {}
        for path in sorted(package_root.rglob('*')):
            if path.is_file() and (path.suffix == '.py' or path.suffix.startswith('.pickle')):
                source_hashes[str(path.relative_to(package_root)).replace('\\', '/')] = hashlib.sha256(path.read_bytes()).hexdigest()
        self.provenance = {'metric_name': 'dimsim', 'metric_version': DIMSIM_VERSION, 'dependencies': versions, 'api': 'dimsim.get_distance([left], [right], pinyin=True)', 'canonical_weights_modified': False, 'package_file_sha256': source_hashes}

    def __call__(self, left: str, right: str) -> float:
        return float(self._get_distance([left], [right], pinyin=True))


## Embedded `evaluation/sampling.py`


In [ ]:
"""Deterministic sampling of unique unordered pairs without quadratic materialization."""
import random
from collections.abc import Iterator, Sequence

def pair_count(item_count: int) -> int:
    """Return the number of unique unordered pairs of ``item_count`` objects."""
    if item_count < 0:
        raise ValueError('item_count must be non-negative')
    return item_count * (item_count - 1) // 2

def pair_from_rank(item_count: int, rank: int) -> tuple[int, int]:
    """Map a rank in lexicographic upper-triangle order to ``(left, right)``."""
    total = pair_count(item_count)
    if rank < 0 or rank >= total:
        raise IndexError(f'pair rank {rank} outside [0, {total})')
    low, high = (0, item_count - 1)
    while low + 1 < high:
        middle = (low + high) // 2
        before = middle * (2 * item_count - middle - 1) // 2
        if before <= rank:
            low = middle
        else:
            high = middle
    left = low
    before = left * (2 * item_count - left - 1) // 2
    right = left + 1 + (rank - before)
    return (left, right)

def iter_all_pairs(item_count: int, *, max_pairs: int) -> Iterator[tuple[int, int]]:
    """Yield all unique pairs, rejecting an unexpectedly large expansion."""
    total = pair_count(item_count)
    if max_pairs < 0:
        raise ValueError('max_pairs must be non-negative')
    if total > max_pairs:
        raise ValueError(f'all-pairs plan has {total} pairs, above cap {max_pairs}')
    for left in range(item_count):
        for right in range(left + 1, item_count):
            yield (left, right)

def sample_pair_indices(item_count: int, sample_count: int, seed: int) -> list[tuple[int, int]]:
    """Uniformly sample unique unordered pairs without constructing all pairs."""
    total = pair_count(item_count)
    if sample_count < 0:
        raise ValueError('sample_count must be non-negative')
    if sample_count > total:
        raise ValueError(f'requested {sample_count} pairs from population of {total}')
    ranks = random.Random(seed).sample(range(total), sample_count)
    return [pair_from_rank(item_count, rank) for rank in sorted(ranks)]

def make_pair_plan(item_ids: Sequence[str], *, sample_count: int | None, seed: int, max_all_pairs: int) -> list[tuple[str, str]]:
    """Return canonical ID pairs from an already stably ordered collection."""
    if len(item_ids) != len(set(item_ids)):
        raise ValueError('item_ids must be unique')
    if list(item_ids) != sorted(item_ids):
        raise ValueError('item_ids must be stably sorted')
    if sample_count is None:
        indices = iter_all_pairs(len(item_ids), max_pairs=max_all_pairs)
    else:
        indices = sample_pair_indices(len(item_ids), sample_count, seed)
    return [(item_ids[left], item_ids[right]) for left, right in indices]


## Embedded `evaluation/bundle.py`


In [ ]:
"""Create and validate immutable, model-independent evaluation bundles.

The on-disk format intentionally uses JSON/JSONL so the preparation stage has
no scientific-runtime dependency. Embeddings are produced by later stages.
"""
import hashlib
import json
import random
import re
import shutil
from collections import defaultdict
from collections.abc import Callable, Iterable
from pathlib import Path
from typing import Any
BUNDLE_SCHEMA_VERSION = 2
PAIRING_POLICY = 'nonzero_nonzero'
_EVALUATION_ID_PATTERN = re.compile('^[A-Za-z0-9](?:[A-Za-z0-9._-]*[A-Za-z0-9])?$')
_WINDOWS_RESERVED_NAME_PATTERN = re.compile('^(?:CON|PRN|AUX|NUL|COM[1-9]|LPT[1-9])(?:\\..*)?$', re.IGNORECASE)
DistanceFunction = Callable[[str, str], float]

def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def _write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')

def _write_jsonl(path: Path, rows: Iterable[dict[str, Any]]) -> None:
    path.write_text(''.join((json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n' for row in rows)), encoding='utf-8')

def _read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding='utf-8'))

def _read_jsonl(path: Path) -> list[dict[str, Any]]:
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line]

def _safe_id(value: str, field: str) -> str:
    if not isinstance(value, str) or not value:
        raise ValueError(f'{field} must be a non-empty string')
    return value

def _validate_evaluation_id(value: Any) -> str:
    if not isinstance(value, str) or _EVALUATION_ID_PATTERN.fullmatch(value) is None or '..' in value or ('/' in value) or ('\\' in value) or (':' in value) or value.endswith(('.', ' ')) or Path(value).is_absolute() or (_WINDOWS_RESERVED_NAME_PATTERN.fullmatch(value) is not None):
        raise ValueError("evaluation_id must be a safe single-directory name using ASCII letters, digits, '.', '_', or '-'")
    return value

def _occurrence_id(sample_id: str, position: int) -> str:
    return f'{sample_id}:{position}'

def _group_id(kind: str, values: tuple[str, ...]) -> str:
    payload = json.dumps([kind, *values], ensure_ascii=False, separators=(',', ':'))
    return f"{kind}:{hashlib.sha256(payload.encode('utf-8')).hexdigest()}"

def _load_source_experiment(root: Path, split: str, sentence_limit: int | None, sentence_seed: int) -> tuple[list[dict], dict, dict, dict]:
    files = {name: root / name for name in ('manifest.json', 'samples.jsonl', 'splits.json', 'vocab.json', 'config.json')}
    missing = [str(path) for path in files.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError(missing)
    manifest = _read_json(files['manifest.json'])
    artifacts = manifest.get('artifacts', {})
    for name in ('samples.jsonl', 'splits.json', 'vocab.json', 'config.json'):
        expected = artifacts.get(name)
        if not isinstance(expected, str) or _sha256(files[name]) != expected:
            raise ValueError(f'source experiment artifact checksum mismatch: {name}')
    splits, vocab = (_read_json(files['splits.json']), _read_json(files['vocab.json']))
    if split not in splits or not isinstance(splits[split], list) or (not splits[split]):
        raise ValueError(f'unknown experiment split: {split}')
    if sentence_limit is not None and (isinstance(sentence_limit, bool) or not isinstance(sentence_limit, int) or sentence_limit <= 0):
        raise ValueError('sentence_limit must be positive or None')
    if any((not isinstance(values, list) for values in splits.values())):
        raise ValueError('source experiment splits must contain ID lists')
    all_split_ids = [sample_id for values in splits.values() for sample_id in values]
    if any((not isinstance(value, str) or not value for value in all_split_ids)) or len(all_split_ids) != len(set(all_split_ids)):
        raise ValueError('source experiment splits are not disjoint unique ID lists')
    selected_ids = sorted(splits[split])
    if sentence_limit is not None:
        if sentence_limit > len(selected_ids):
            raise ValueError(f'sentence_limit {sentence_limit} exceeds split size {len(selected_ids)}')
        selected_ids = sorted(random.Random(sentence_seed).sample(selected_ids, sentence_limit))
    sample_rows = _read_jsonl(files['samples.jsonl'])
    samples = {row['sample_id']: row for row in sample_rows}
    if len(samples) != len(sample_rows):
        raise ValueError('duplicate sample IDs in source experiment')
    try:
        selected = [samples[sample_id] for sample_id in selected_ids]
    except KeyError as error:
        raise ValueError(f'split references missing sample: {error.args[0]}') from error
    hashes = {name: _sha256(files[name]) for name in ('manifest.json', 'samples.jsonl', 'splits.json', 'vocab.json', 'config.json')}
    return (selected, vocab, manifest, hashes)

def _build_occurrences(samples: list[dict], vocab: dict) -> list[dict]:
    acoustic_tokens = vocab.get('acoustic_tokens')
    labels = vocab.get('labels', {})
    if not isinstance(acoustic_tokens, list) or any((not isinstance(x, str) for x in acoustic_tokens)):
        raise ValueError('invalid acoustic token vocabulary')
    component_names = [labels.get(name) for name in ('initials', 'finals', 'tones')]
    if any((not isinstance(values, list) or any((not isinstance(value, str) for value in values)) for values in component_names)):
        raise ValueError('invalid component label vocabulary')
    rows = []
    for sample in sorted(samples, key=lambda row: row['sample_id']):
        sample_id, text = (_safe_id(sample.get('sample_id'), 'sample_id'), sample.get('text'))
        acoustic_ids, components = (sample.get('acoustic_ids'), sample.get('component_labels'))
        if not isinstance(text, str) or not len(text) == len(acoustic_ids or []) == len(components or []):
            raise ValueError(f'source sample shape mismatch: {sample_id}')
        for position, character in enumerate(text):
            acoustic_id = acoustic_ids[position]
            component_ids = components[position]
            if not isinstance(acoustic_id, int) or acoustic_id < 0 or acoustic_id >= len(acoustic_tokens):
                raise ValueError(f'invalid acoustic ID in sample {sample_id}')
            if not isinstance(component_ids, list) or len(component_ids) != 3:
                raise ValueError(f'invalid component labels in sample {sample_id}')
            if any((isinstance(value, bool) or not isinstance(value, int) or value < 0 or (value >= len(component_names[i])) for i, value in enumerate(component_ids))):
                raise ValueError(f'component ID outside vocabulary in sample {sample_id}')
            try:
                initial, final, tone = (component_names[i][value] for i, value in enumerate(component_ids))
            except (IndexError, TypeError) as error:
                raise ValueError(f'component ID outside vocabulary in sample {sample_id}') from error
            pinyin = acoustic_tokens[acoustic_id]
            rows.append({'occurrence_id': _occurrence_id(sample_id, position), 'sample_id': sample_id, 'character_position': position, 'character': character, 'pinyin': pinyin, 'initial': initial, 'final': final, 'tone': tone})
    return sorted(rows, key=lambda row: row['occurrence_id'])

def _build_groups(occurrences: list[dict]) -> tuple[list[dict], dict[str, dict]]:
    members: dict[str, list[str]] = defaultdict(list)
    metadata: dict[str, dict] = {}
    for row in occurrences:
        definitions = (('character_pinyin', (row['character'], row['pinyin'])), ('syllable', (row['pinyin'],)))
        for kind, keys in definitions:
            group_id = _group_id(kind, keys)
            members[group_id].append(row['occurrence_id'])
            metadata[group_id] = {'group_id': group_id, 'group_kind': kind, 'keys': list(keys), 'pinyin': row['pinyin'], 'initial': row['initial'], 'final': row['final'], 'tone': row['tone']}
    groups = []
    for group_id in sorted(members):
        groups.append({**metadata[group_id], 'occurrence_ids': sorted(members[group_id]), 'occurrence_count': len(members[group_id])})
    return (groups, {row['group_id']: row for row in groups})

def _pair_rows(ids: list[str], metadata: dict[str, dict], *, sample_count: int | None, seed: int, max_all_pairs: int, distance_fn: DistanceFunction | None) -> list[dict]:
    rows = []
    for left_id, right_id in make_pair_plan(ids, sample_count=sample_count, seed=seed, max_all_pairs=max_all_pairs):
        left, right = (metadata[left_id], metadata[right_id])
        status, distance = ('not_computed', None)
        if distance_fn is not None:
            try:
                distance = float(distance_fn(left['pinyin'], right['pinyin']))
                if distance != distance or distance in (float('inf'), float('-inf')):
                    raise ValueError('distance is not finite')
                status = 'ok'
            except Exception as error:
                status = f'invalid:{type(error).__name__}'
                distance = None
        equal_initial = left['initial'] == right['initial']
        equal_final = left['final'] == right['final']
        equal_tone = left['tone'] == right['tone']
        pair_id = hashlib.sha256(json.dumps([left_id, right_id], ensure_ascii=False, separators=(',', ':')).encode()).hexdigest()
        rows.append({'pair_id': pair_id, 'left_id': left_id, 'right_id': right_id, 'left_pinyin': left['pinyin'], 'right_pinyin': right['pinyin'], 'dimsim_distance': distance, 'distance_status': status, 'equal_initial': equal_initial, 'equal_final': equal_final, 'equal_tone': equal_tone, 'overlap_mask': f'{int(equal_initial)}{int(equal_final)}{int(equal_tone)}', 'left_zero_initial': left['initial'] == '', 'right_zero_initial': right['initial'] == '', 'no_zero_initial_diagnostic': left['initial'] != '' and right['initial'] != ''})
    return rows

def prepare_evaluation_bundle(evaluation_id: str, *, experiment_path: str | Path, output_root: str | Path, split: str='test', sentence_limit: int | None=None, contextual_pair_count: int=12000, sentence_seed: int=42, sampling_seed: int=42, character_pair_count: int=12000, syllable_pair_count: int | None=None, max_all_pairs: int=1000000, distance_fn: DistanceFunction | None=None, metric_provenance: dict[str, Any] | None=None, code_provenance: dict[str, str] | None=None) -> Path:
    """Create an immutable evaluation bundle and return its path."""
    evaluation_id = _validate_evaluation_id(evaluation_id)
    numeric = {'contextual_pair_count': contextual_pair_count, 'sentence_seed': sentence_seed, 'sampling_seed': sampling_seed, 'character_pair_count': character_pair_count, 'max_all_pairs': max_all_pairs}
    if any((isinstance(value, bool) or not isinstance(value, int) for value in numeric.values())):
        raise TypeError('pair counts, caps, and seeds must be integers')
    if contextual_pair_count < 0 or character_pair_count < 0 or max_all_pairs <= 0:
        raise ValueError('pair counts must be non-negative and max_all_pairs positive')
    if syllable_pair_count is not None and (isinstance(syllable_pair_count, bool) or not isinstance(syllable_pair_count, int) or syllable_pair_count < 0):
        raise ValueError('syllable_pair_count must be a non-negative integer or None')
    if distance_fn is None or not callable(distance_fn):
        raise ValueError('an explicit validated distance provider is required')
    if not isinstance(metric_provenance, dict) or not isinstance(metric_provenance.get('metric_name'), str):
        raise ValueError('metric_provenance with an explicit metric_name is required')
    metric_name = metric_provenance['metric_name']
    if metric_name == 'dimsim':
        if not isinstance(distance_fn, DimsimDistance) or metric_provenance != distance_fn.provenance:
            raise ValueError('DIMSIM bundles require the validated pinned DimsimDistance provider and its provenance')
    elif metric_name != 'synthetic':
        raise ValueError('only the pinned DIMSIM provider or an explicitly labelled synthetic test provider is supported')
    if not isinstance(code_provenance, dict) or not code_provenance or any((not isinstance(key, str) or not isinstance(value, str) or len(value) != 64 or any((character not in '0123456789abcdef' for character in value)) for key, value in code_provenance.items())):
        raise ValueError('explicit non-empty preparation code_provenance hashes are required')
    experiment_root, root = (Path(experiment_path).resolve(), Path(output_root).resolve())
    target = (root / evaluation_id).resolve()
    if target.parent != root:
        raise ValueError('evaluation path escapes output_root')
    if target == experiment_root or experiment_root in target.parents:
        raise ValueError('evaluation output must not be inside the source experiment')
    if target.exists():
        raise FileExistsError(f'evaluation bundle already exists: {target}')
    samples, vocab, source_manifest, source_hashes = _load_source_experiment(experiment_root, split, sentence_limit, sentence_seed)
    occurrences = _build_occurrences(samples, vocab)
    groups, group_by_id = _build_groups(occurrences)
    occurrence_by_id = {row['occurrence_id']: row for row in occurrences}
    eligible_ids = {'contextual': sorted((row['occurrence_id'] for row in occurrences if isinstance(row['initial'], str) and row['initial'])), 'character_pinyin': sorted((row['group_id'] for row in groups if row['group_kind'] == 'character_pinyin' and isinstance(row['initial'], str) and row['initial'])), 'syllable': sorted((row['group_id'] for row in groups if row['group_kind'] == 'syllable' and isinstance(row['initial'], str) and row['initial']))}
    contextual = _pair_rows(eligible_ids['contextual'], occurrence_by_id, sample_count=contextual_pair_count, seed=sampling_seed, max_all_pairs=max_all_pairs, distance_fn=distance_fn)
    aggregate_plans = {}
    pair_counts = {'character_pinyin': character_pair_count, 'syllable': syllable_pair_count}
    for offset, kind in enumerate(('character_pinyin', 'syllable'), start=1):
        aggregate_plans[kind] = _pair_rows(eligible_ids[kind], group_by_id, sample_count=pair_counts[kind], seed=sampling_seed + offset, max_all_pairs=max_all_pairs, distance_fn=distance_fn)
    target.mkdir(parents=True)
    try:
        (target / 'pairs').mkdir()
        selected_sentences = [{'sample_id': row['sample_id'], 'text': row['text']} for row in sorted(samples, key=lambda x: x['sample_id'])]
        _write_jsonl(target / 'sentences.jsonl', selected_sentences)
        _write_jsonl(target / 'occurrences.jsonl', occurrences)
        _write_jsonl(target / 'groups.jsonl', groups)
        _write_jsonl(target / 'pairs/contextual.jsonl', contextual)
        for kind, rows in aggregate_plans.items():
            _write_jsonl(target / f'pairs/{kind}.jsonl', rows)
        object_counts = {'contextual': len(occurrences), 'character_pinyin': sum((row['group_kind'] == 'character_pinyin' for row in groups)), 'syllable': sum((row['group_kind'] == 'syllable' for row in groups))}
        all_plans = {'contextual': contextual, **aggregate_plans}
        coverage = {'sentences': len(samples), 'occurrences': len(occurrences), 'groups': {kind: object_counts[kind] for kind in ('character_pinyin', 'syllable')}, 'levels': {}}
        for level, rows in all_plans.items():
            statuses: dict[str, int] = {}
            for row in rows:
                statuses[row['distance_status']] = statuses.get(row['distance_status'], 0) + 1
            original_count = object_counts[level]
            eligible_count = len(eligible_ids[level])
            original_pairs, eligible_pairs = (pair_count(original_count), pair_count(eligible_count))
            coverage['levels'][level] = {'objects': {'original': original_count, 'eligible': eligible_count, 'excluded': original_count - eligible_count, 'excluded_by_reason': {'zero_initial': original_count - eligible_count}}, 'unordered_pair_population': {'original': original_pairs, 'eligible': eligible_pairs, 'excluded': original_pairs - eligible_pairs, 'excluded_by_reason': {'zero_initial': original_pairs - eligible_pairs}}, 'selection_count': len(rows), 'distance_status_counts': dict(sorted(statuses.items()))}
        _write_json(target / 'coverage.json', coverage)
        config = {'schema_version': BUNDLE_SCHEMA_VERSION, 'evaluation_id': evaluation_id, 'split': split, 'sentence_limit': sentence_limit, 'contextual_pair_count': contextual_pair_count, 'sentence_seed': sentence_seed, 'sampling_seed': sampling_seed, 'character_pair_count': character_pair_count, 'syllable_pair_count': syllable_pair_count, 'max_all_pairs': max_all_pairs, 'pairing_policy': PAIRING_POLICY}
        _write_json(target / 'config.json', config)
        artifacts = {str(path.relative_to(target)).replace('\\', '/'): _sha256(path) for path in target.rglob('*') if path.is_file()}
        manifest = {'schema_version': BUNDLE_SCHEMA_VERSION, 'evaluation_id': evaluation_id, 'pairing_policy': PAIRING_POLICY, 'source_experiment': {'path': str(experiment_root), 'experiment_id': source_manifest.get('experiment_id'), 'hashes': source_hashes}, 'metric': metric_provenance, 'code_provenance': code_provenance, 'counts': {'sentences': len(samples), 'occurrences': len(occurrences), 'character_pinyin_groups': sum((row['group_kind'] == 'character_pinyin' for row in groups)), 'syllable_groups': sum((row['group_kind'] == 'syllable' for row in groups)), 'pairs': {'contextual': len(contextual), **{kind: len(rows) for kind, rows in aggregate_plans.items()}}}, 'artifacts': artifacts}
        _write_json(target / 'manifest.json', manifest)
    except Exception:
        shutil.rmtree(target)
        raise
    return target

def load_evaluation_bundle(path: str | Path) -> dict[str, Any]:
    """Load a bundle after verifying every recorded artifact checksum."""
    root = Path(path).resolve()
    manifest = _read_json(root / 'manifest.json')
    if manifest.get('schema_version') != BUNDLE_SCHEMA_VERSION:
        raise ValueError('unsupported evaluation bundle schema')
    config = _read_json(root / 'config.json')
    if config.get('schema_version') != BUNDLE_SCHEMA_VERSION:
        raise ValueError('unsupported evaluation config schema')
    if manifest.get('pairing_policy') != PAIRING_POLICY or config.get('pairing_policy') != PAIRING_POLICY:
        raise ValueError('unsupported evaluation pairing policy')
    manifest_evaluation_id = _validate_evaluation_id(manifest.get('evaluation_id'))
    config_evaluation_id = _validate_evaluation_id(config.get('evaluation_id'))
    if manifest_evaluation_id != config_evaluation_id:
        raise ValueError('invalid or inconsistent evaluation_id')
    required = {'config.json', 'coverage.json', 'sentences.jsonl', 'occurrences.jsonl', 'groups.jsonl', 'pairs/contextual.jsonl', 'pairs/character_pinyin.jsonl', 'pairs/syllable.jsonl'}
    artifacts = manifest.get('artifacts', {})
    if not required.issubset(artifacts):
        raise ValueError('evaluation manifest is missing required artifacts')
    for relative, expected in artifacts.items():
        artifact = (root / relative).resolve()
        if root not in artifact.parents or not artifact.is_file() or _sha256(artifact) != expected:
            raise ValueError(f'evaluation artifact checksum mismatch: {relative}')
    coverage = _read_json(root / 'coverage.json')
    occurrences, groups = (_read_jsonl(root / 'occurrences.jsonl'), _read_jsonl(root / 'groups.jsonl'))
    occurrence_ids = [row.get('occurrence_id') for row in occurrences]
    group_ids = [row.get('group_id') for row in groups]
    if len(occurrence_ids) != len(set(occurrence_ids)) or any((not isinstance(value, str) for value in occurrence_ids)):
        raise ValueError('evaluation occurrence IDs are invalid or duplicated')
    if len(group_ids) != len(set(group_ids)) or any((not isinstance(value, str) for value in group_ids)):
        raise ValueError('evaluation group IDs are invalid or duplicated')
    if any((not isinstance(row.get('initial'), str) for row in occurrences + groups)):
        raise ValueError('evaluation initial metadata must be strings')
    pairs = {kind: _read_jsonl(root / f'pairs/{kind}.jsonl') for kind in ('contextual', 'character_pinyin', 'syllable')}
    occurrence_id_set = set(occurrence_ids)
    valid_ids = {'contextual': occurrence_id_set, 'character_pinyin': {row['group_id'] for row in groups if row.get('group_kind') == 'character_pinyin'}, 'syllable': {row['group_id'] for row in groups if row.get('group_kind') == 'syllable'}}
    metadata = {'contextual': {row['occurrence_id']: row for row in occurrences}, 'character_pinyin': {row['group_id']: row for row in groups if row.get('group_kind') == 'character_pinyin'}, 'syllable': {row['group_id']: row for row in groups if row.get('group_kind') == 'syllable'}}
    for group in groups:
        members = group.get('occurrence_ids')
        if not isinstance(members, list) or not members or members != sorted(members) or (len(members) != len(set(members))) or (group.get('occurrence_count') != len(members)) or any((member not in occurrence_id_set for member in members)):
            raise ValueError(f"invalid group membership: {group.get('group_id')}")
    for kind, rows in pairs.items():
        pair_ids = [row.get('pair_id') for row in rows]
        if len(pair_ids) != len(set(pair_ids)) or any((not isinstance(value, str) for value in pair_ids)):
            raise ValueError(f'{kind} pair IDs are invalid or duplicated')
        if any((row.get('left_id') not in valid_ids[kind] or row.get('right_id') not in valid_ids[kind] for row in rows)):
            raise ValueError(f'{kind} pair references an unknown object')
        endpoints = [(row['left_id'], row['right_id']) for row in rows]
        if len(endpoints) != len(set(endpoints)) or any((left >= right for left, right in endpoints)):
            raise ValueError(f'{kind} pairs are duplicated or non-canonical')
        for row in rows:
            expected = hashlib.sha256(json.dumps([row['left_id'], row['right_id']], ensure_ascii=False, separators=(',', ':')).encode()).hexdigest()
            if row['pair_id'] != expected:
                raise ValueError(f'{kind} pair ID does not match endpoints')
            left_zero = metadata[kind][row['left_id']].get('initial') == ''
            right_zero = metadata[kind][row['right_id']].get('initial') == ''
            if left_zero or right_zero:
                raise ValueError(f'{kind} pair violates {PAIRING_POLICY} policy')
            if row.get('left_zero_initial') is not False or row.get('right_zero_initial') is not False or row.get('no_zero_initial_diagnostic') is not True:
                raise ValueError(f'{kind} pair has inconsistent zero-initial flags')
        original_count = len(valid_ids[kind])
        eligible_count = sum((isinstance(metadata[kind][object_id].get('initial'), str) and bool(metadata[kind][object_id]['initial']) for object_id in valid_ids[kind]))
        original_pairs, eligible_pairs = (pair_count(original_count), pair_count(eligible_count))
        statuses: dict[str, int] = {}
        for row in rows:
            status = row.get('distance_status')
            statuses[status] = statuses.get(status, 0) + 1
        expected_coverage = {'objects': {'original': original_count, 'eligible': eligible_count, 'excluded': original_count - eligible_count, 'excluded_by_reason': {'zero_initial': original_count - eligible_count}}, 'unordered_pair_population': {'original': original_pairs, 'eligible': eligible_pairs, 'excluded': original_pairs - eligible_pairs, 'excluded_by_reason': {'zero_initial': original_pairs - eligible_pairs}}, 'selection_count': len(rows), 'distance_status_counts': dict(sorted(statuses.items()))}
        if coverage.get('levels', {}).get(kind) != expected_coverage:
            raise ValueError(f'{kind} coverage metadata is inconsistent')
    expected_counts = {'sentences': len(_read_jsonl(root / 'sentences.jsonl')), 'occurrences': len(occurrences), 'character_pinyin_groups': len(valid_ids['character_pinyin']), 'syllable_groups': len(valid_ids['syllable']), 'pairs': {kind: len(rows) for kind, rows in pairs.items()}}
    if manifest.get('counts') != expected_counts:
        raise ValueError('evaluation manifest counts are inconsistent')
    return {'root': root, 'manifest': manifest, 'config': config, 'coverage': coverage, 'sentences': _read_jsonl(root / 'sentences.jsonl'), 'occurrences': occurrences, 'groups': groups, 'pairs': pairs}


## Embedded `evaluation/representations.py`


In [ ]:
"""Storage, aggregation, and random baselines for aligned representation matrices."""
import hashlib
import json
from pathlib import Path
from typing import Any
import numpy as np

def validate_matrix(ids: list[str], matrix: np.ndarray, *, require_nonzero: bool=True) -> np.ndarray:
    values = np.asarray(matrix)
    if values.ndim != 2 or values.shape[0] != len(ids) or len(ids) != len(set(ids)):
        raise ValueError('representation IDs and matrix shape/uniqueness disagree')
    if ids != sorted(ids):
        raise ValueError('representation IDs must be stably sorted')
    if not np.isfinite(values).all():
        raise ValueError('representation matrix contains non-finite values')
    if require_nonzero and np.any(np.linalg.norm(values.astype(np.float64), axis=1) <= 1e-12):
        raise ValueError('representation matrix contains a zero-norm row')
    return values

def aggregate_occurrences(occurrence_ids: list[str], hidden: np.ndarray, groups: list[dict[str, Any]], group_kind: str) -> tuple[list[str], np.ndarray, list[int]]:
    """Mean raw hidden states over every occurrence member of each group."""
    values = validate_matrix(occurrence_ids, hidden)
    positions = {occurrence_id: index for index, occurrence_id in enumerate(occurrence_ids)}
    selected = sorted((row for row in groups if row.get('group_kind') == group_kind), key=lambda row: row['group_id'])
    group_ids, means, counts = ([], [], [])
    for group in selected:
        members = group.get('occurrence_ids')
        if not isinstance(members, list) or not members or len(members) != len(set(members)):
            raise ValueError(f"invalid membership for group {group.get('group_id')}")
        try:
            indices = [positions[member] for member in members]
        except KeyError as error:
            raise ValueError(f'group references missing occurrence {error.args[0]}') from error
        group_ids.append(group['group_id'])
        means.append(values[indices].astype(np.float64).mean(axis=0))
        counts.append(len(indices))
    matrix = np.asarray(means, dtype=np.float32)
    validate_matrix(group_ids, matrix)
    return (group_ids, matrix, counts)

def random_occurrence_baseline(occurrence_ids: list[str], *, dimension: int, seed: int) -> np.ndarray:
    if dimension <= 0 or isinstance(dimension, bool):
        raise ValueError('dimension must be a positive integer')
    if occurrence_ids != sorted(occurrence_ids) or len(occurrence_ids) != len(set(occurrence_ids)):
        raise ValueError('occurrence IDs must be unique and sorted')
    matrix = np.random.default_rng(seed).standard_normal((len(occurrence_ids), dimension)).astype(np.float32)
    return validate_matrix(occurrence_ids, matrix)

def save_representations(output_dir: str | Path, *, occurrence_ids: list[str], hidden: np.ndarray, groups: list[dict[str, Any]], provenance: dict[str, Any]) -> Path:
    target = Path(output_dir).resolve()
    if target.exists():
        raise FileExistsError(f'representation output already exists: {target}')
    validate_matrix(occurrence_ids, hidden)
    aggregates = {kind: aggregate_occurrences(occurrence_ids, hidden, groups, kind) for kind in ('character_pinyin', 'syllable')}
    target.mkdir(parents=True)
    (target / 'occurrence_ids.json').write_text(json.dumps(occurrence_ids, ensure_ascii=False) + '\n', encoding='utf-8')
    np.save(target / 'occurrence_hidden.npy', np.asarray(hidden, dtype=np.float32), allow_pickle=False)
    for kind, (ids, matrix, counts) in aggregates.items():
        (target / f'{kind}_ids.json').write_text(json.dumps(ids, ensure_ascii=False) + '\n', encoding='utf-8')
        (target / f'{kind}_counts.json').write_text(json.dumps(counts) + '\n', encoding='utf-8')
        np.save(target / f'{kind}_hidden.npy', matrix, allow_pickle=False)
    artifacts = {}
    for path in target.iterdir():
        if path.is_file():
            artifacts[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
    manifest = {'schema_version': 1, 'provenance': provenance, 'artifacts': artifacts}
    (target / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    return target


## Embedded `model.py`


In [ ]:
"""Canonical encoder shared by all training conditions."""
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class ClassifierHead(nn.Module):

    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int, dropout: float=0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.dropout(self.act(self.fc1(x))))

class Encoder(nn.Module):
    """Byte-for-byte state-dict-compatible extraction of the pure encoder."""

    def __init__(self, vocab_size_tokens: int, embedding_dim: int, hidden_dim: int, num_initials: int, num_finals: int, num_tones: int, num_layers: int=4, dropout: float=0.3, mlp_hidden_dim: int=256):
        super().__init__()
        self.embedding_bag = nn.EmbeddingBag(vocab_size_tokens, embedding_dim, mode='mean')
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, bidirectional=True, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.initial_head = ClassifierHead(hidden_dim * 2, mlp_hidden_dim, num_initials, dropout)
        self.final_head = ClassifierHead(hidden_dim * 2, mlp_hidden_dim, num_finals, dropout)
        self.tone_head = ClassifierHead(hidden_dim * 2, mlp_hidden_dim, num_tones, dropout)

    def forward(self, token_ids, token_offsets, char_lengths, return_hidden=False):
        char_emb = self.embedding_bag(token_ids, token_offsets)
        padded = nn.utils.rnn.pad_sequence(torch.split(char_emb, char_lengths.tolist()), batch_first=True)
        packed = pack_padded_sequence(padded, char_lengths.cpu(), batch_first=True, enforce_sorted=False)
        char_out, _ = pad_packed_sequence(self.lstm(packed)[0], batch_first=True)
        char_out = self.dropout(char_out)
        result = (self.initial_head(char_out), self.final_head(char_out), self.tone_head(char_out))
        return (*result, char_out) if return_hidden else result

def build_encoder(config: dict, vocab: dict) -> Encoder:
    arch = config['architecture']
    acoustic_dim = int(config['acoustic_dim'])
    if 2 * int(arch['hidden_dim']) != acoustic_dim:
        raise ValueError('2 * hidden_dim must equal acoustic_dim for normalized-MSE alignment')
    labels = vocab['labels']
    return Encoder(len(vocab['wubi_tokens']), int(arch['embedding_dim']), int(arch['hidden_dim']), len(labels['initials']), len(labels['finals']), len(labels['tones']), int(arch['num_layers']), float(arch['dropout']), int(arch['mlp_hidden_dim']))


## Embedded `evaluation/encoders.py`


In [ ]:
"""Strict adapters that align encoder outputs to model-independent occurrences."""
import hashlib
import json
from pathlib import Path
import numpy as np
import torch

def _sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def extract_bilstm(bundle_path: str | Path, *, experiment_path: str | Path, checkpoint_path: str | Path, output_dir: str | Path, device: str='cpu', batch_size: int=32) -> Path:
    """Extract full-sentence BiLSTM outputs and save exact occurrence alignment."""
    if batch_size <= 0 or isinstance(batch_size, bool):
        raise ValueError('batch_size must be a positive integer')
    bundle = load_evaluation_bundle(bundle_path)
    experiment_root, checkpoint_file = (Path(experiment_path).resolve(), Path(checkpoint_path).resolve())
    source_hashes = bundle['manifest']['source_experiment']['hashes']
    for name in ('manifest.json', 'config.json', 'vocab.json', 'samples.jsonl'):
        path = experiment_root / name
        if not path.is_file() or _sha256(path) != source_hashes.get(name):
            raise ValueError(f'experiment does not match evaluation source hash: {name}')
    config = json.loads((experiment_root / 'config.json').read_text(encoding='utf-8'))
    vocab = json.loads((experiment_root / 'vocab.json').read_text(encoding='utf-8'))
    checkpoint = torch.load(checkpoint_file, map_location='cpu', weights_only=True)
    expected = {'manifest_sha256': source_hashes['manifest.json'], 'config_sha256': source_hashes['config.json'], 'vocab_sha256': source_hashes['vocab.json']}
    if any((checkpoint.get(key) != value for key, value in expected.items())):
        raise ValueError('checkpoint provenance does not match evaluation experiment')
    model = build_encoder(config, vocab)
    model.load_state_dict(checkpoint['state_dict'], strict=True)
    model.to(torch.device(device)).eval()
    sample_rows = [json.loads(line) for line in (experiment_root / 'samples.jsonl').read_text(encoding='utf-8').splitlines()]
    samples = {row['sample_id']: row for row in sample_rows}
    occurrence_by_sample: dict[str, list[dict]] = {}
    for occurrence in bundle['occurrences']:
        occurrence_by_sample.setdefault(occurrence['sample_id'], []).append(occurrence)
    sentence_ids = [row['sample_id'] for row in bundle['sentences']]
    if set(sentence_ids) != set(occurrence_by_sample) or len(sentence_ids) != len(set(sentence_ids)):
        raise ValueError('bundle sentences and occurrence samples disagree')
    extracted: dict[str, np.ndarray] = {}
    with torch.no_grad():
        for start in range(0, len(sentence_ids), batch_size):
            batch_ids = sentence_ids[start:start + batch_size]
            selected = [samples[sample_id] for sample_id in batch_ids]
            token_ids, offsets, cursor = ([], [], 0)
            for sample in selected:
                token_ids.extend(sample['token_ids'])
                offsets.extend((cursor + span[0] for span in sample['character_spans']))
                cursor += len(sample['token_ids'])
            lengths = torch.tensor([len(sample['character_spans']) for sample in selected], dtype=torch.long)
            outputs = model(torch.tensor(token_ids, dtype=torch.long, device=device), torch.tensor(offsets, dtype=torch.long, device=device), lengths, return_hidden=True)[-1].detach().cpu().numpy()
            for row_index, (sample_id, sample) in enumerate(zip(batch_ids, selected)):
                expected_positions = sorted(occurrence_by_sample[sample_id], key=lambda row: row['character_position'])
                if [row['character_position'] for row in expected_positions] != list(range(len(sample['text']))) or any((row['character'] != sample['text'][row['character_position']] for row in expected_positions)):
                    raise ValueError(f'bundle occurrences do not exactly cover source sentence {sample_id}')
                for occurrence in expected_positions:
                    extracted[occurrence['occurrence_id']] = outputs[row_index, occurrence['character_position']]
    occurrence_ids = sorted((row['occurrence_id'] for row in bundle['occurrences']))
    if set(extracted) != set(occurrence_ids):
        raise ValueError('encoder extraction did not produce every expected occurrence')
    hidden = np.stack([extracted[occurrence_id] for occurrence_id in occurrence_ids]).astype(np.float32)
    validate_matrix(occurrence_ids, hidden)
    provenance = {'adapter': 'phonetics.evaluation.encoders.extract_bilstm', 'device': str(device), 'checkpoint_path': str(checkpoint_file), 'checkpoint_sha256': _sha256(checkpoint_file), 'bundle_manifest_sha256': _sha256(Path(bundle_path).resolve() / 'manifest.json'), 'experiment_hashes': expected, 'hidden_dimension': int(hidden.shape[1])}
    return save_representations(output_dir, occurrence_ids=occurrence_ids, hidden=hidden, groups=bundle['groups'], provenance=provenance)


## Embedded `evaluation/analysis.py`


In [ ]:
"""RSA-style cosine analysis and strict paired comparison of encoder runs."""
import hashlib
import json
import math
from pathlib import Path
from typing import Any
import numpy as np
LEVELS = ('contextual', 'character_pinyin', 'syllable')
BIN_EDGES = tuple(range(0, 121, 10))

def _sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def distance_bin(distance: float | None) -> tuple[str, str]:
    """Return ``(status, label)`` with an explicit unbounded overflow bin."""
    if distance is None or not isinstance(distance, (int, float)) or (not math.isfinite(float(distance))):
        return ('invalid_nonfinite', 'invalid')
    value = float(distance)
    if value < 0:
        return ('invalid_negative', 'invalid')
    if value >= 120:
        return ('ok', '[120,+inf)')
    lower = int(value // 10) * 10
    return ('ok', f'[{lower},{lower + 10})')

def _rankdata(values: np.ndarray) -> np.ndarray:
    order = np.argsort(values, kind='mergesort')
    ranks = np.empty(len(values), dtype=np.float64)
    position = 0
    while position < len(values):
        end = position + 1
        while end < len(values) and values[order[end]] == values[order[position]]:
            end += 1
        ranks[order[position:end]] = (position + end - 1) / 2.0 + 1.0
        position = end
    return ranks

def spearman_result(distances: list[float], dissimilarities: list[float]) -> dict[str, Any]:
    if len(distances) != len(dissimilarities):
        raise ValueError('Spearman inputs have different lengths')
    count = len(distances)
    if count < 2:
        return {'rho': None, 'valid_pair_count': count, 'undefined_reason': 'fewer_than_two_pairs'}
    left, right = (np.asarray(distances, dtype=np.float64), np.asarray(dissimilarities, dtype=np.float64))
    if not np.isfinite(left).all() or not np.isfinite(right).all():
        raise ValueError('Spearman inputs must be finite after validity filtering')
    left_rank, right_rank = (_rankdata(left), _rankdata(right))
    if np.ptp(left_rank) == 0 or np.ptp(right_rank) == 0:
        reason = 'constant_distance' if np.ptp(left_rank) == 0 else 'constant_hidden_dissimilarity'
        return {'rho': None, 'valid_pair_count': count, 'undefined_reason': reason}
    rho = float(np.corrcoef(left_rank, right_rank)[0, 1])
    return {'rho': rho, 'valid_pair_count': count, 'undefined_reason': None}

def cosine_rows(pair_rows: list[dict], object_ids: list[str], matrix: np.ndarray) -> list[dict]:
    values = validate_matrix(object_ids, matrix)
    index = {object_id: position for position, object_id in enumerate(object_ids)}
    result = []
    for pair in pair_rows:
        try:
            left, right = (values[index[pair['left_id']]], values[index[pair['right_id']]])
        except KeyError as error:
            raise ValueError(f'pair references missing representation {error.args[0]}') from error
        cosine = float(np.dot(left.astype(np.float64), right.astype(np.float64)) / (np.linalg.norm(left.astype(np.float64)) * np.linalg.norm(right.astype(np.float64))))
        cosine = min(1.0, max(-1.0, cosine))
        bin_status, label = distance_bin(pair.get('dimsim_distance'))
        metric_status = pair.get('distance_status', 'missing_status')
        analysis_status = 'ok' if metric_status == 'ok' and bin_status == 'ok' else metric_status if metric_status != 'ok' else bin_status
        result.append({**pair, 'cosine_similarity': cosine, 'hidden_dissimilarity': 1.0 - cosine, 'analysis_status': analysis_status, 'distance_bin': label})
    return result

def summarize_rows(rows: list[dict]) -> dict[str, Any]:
    valid = [row for row in rows if row['analysis_status'] == 'ok']

    def correlation(selected: list[dict]) -> dict[str, Any]:
        return spearman_result([float(row['dimsim_distance']) for row in selected], [float(row['hidden_dissimilarity']) for row in selected])
    strata = {mask: correlation([row for row in valid if row.get('overlap_mask') == mask]) for mask in (f'{value:03b}' for value in range(8))}
    strata['no_zero_initial'] = correlation([row for row in valid if row.get('no_zero_initial_diagnostic') is True])
    bins = []
    labels = [f'[{lower},{lower + 10})' for lower in range(0, 120, 10)] + ['[120,+inf)']
    for label in labels:
        selected = [row for row in valid if row['distance_bin'] == label]
        distances = [float(row['dimsim_distance']) for row in selected]
        cosines = [float(row['cosine_similarity']) for row in selected]
        bins.append({'bin': label, 'count': len(selected), 'distance_min': min(distances) if distances else None, 'distance_max': max(distances) if distances else None, 'cosine_mean': float(np.mean(cosines)) if cosines else None})
    status_counts: dict[str, int] = {}
    for row in rows:
        status_counts[row['analysis_status']] = status_counts.get(row['analysis_status'], 0) + 1
    return {'pair_count': len(rows), 'status_counts': dict(sorted(status_counts.items())), 'overall': correlation(valid), 'strata': strata, 'bins': bins, 'uncertainty': {'status': 'not_estimated_single_seed', 'interpretation': 'fixed-pair descriptive statistic; no population or training-run confidence interval', 'iid_pair_bootstrap': False, 'p_values_reported': False}}

def _load_representation_level(root: Path, level: str) -> tuple[list[str], np.ndarray]:
    prefix = 'occurrence' if level == 'contextual' else level
    ids = json.loads((root / f'{prefix}_ids.json').read_text(encoding='utf-8'))
    matrix = np.load(root / f'{prefix}_hidden.npy', allow_pickle=False)
    return (ids, matrix)

def _plot_bins(summary: dict, path: Path, title: str) -> None:
    import matplotlib.pyplot as plt
    populated = [row for row in summary['bins'] if row['count']]
    figure, axis = plt.subplots(figsize=(10, 4))
    axis.plot([row['bin'] for row in populated], [row['cosine_mean'] for row in populated], marker='o')
    for index, row in enumerate(populated):
        axis.annotate(f"n={row['count']}", (index, row['cosine_mean']), xytext=(0, 6), textcoords='offset points', ha='center', fontsize=7)
    axis.set(title=title, xlabel='DIMSIM distance bin', ylabel='Mean raw cosine')
    axis.tick_params(axis='x', rotation=45)
    figure.tight_layout()
    figure.savefig(path, dpi=150)
    plt.close(figure)

def analyze_representations(bundle_path: str | Path, representations_dir: str | Path, *, output_dir: str | Path, run_label: str) -> Path:
    bundle, representation_root = (load_evaluation_bundle(bundle_path), Path(representations_dir).resolve())
    representation_manifest = json.loads((representation_root / 'manifest.json').read_text(encoding='utf-8'))
    required_representation_artifacts = {'occurrence_ids.json', 'occurrence_hidden.npy'}
    for level in ('character_pinyin', 'syllable'):
        required_representation_artifacts.update({f'{level}_ids.json', f'{level}_counts.json', f'{level}_hidden.npy'})
    representation_artifacts = representation_manifest.get('artifacts', {})
    if not required_representation_artifacts.issubset(representation_artifacts):
        raise ValueError('representation manifest is missing required artifacts')
    for name, digest in representation_artifacts.items():
        artifact = (representation_root / name).resolve()
        if representation_root not in artifact.parents or not artifact.is_file() or _sha256(artifact) != digest:
            raise ValueError(f'representation artifact checksum mismatch: {name}')
    bundle_manifest_sha = _sha256(Path(bundle_path).resolve() / 'manifest.json')
    if representation_manifest.get('provenance', {}).get('bundle_manifest_sha256') != bundle_manifest_sha:
        raise ValueError('representations were not extracted from this evaluation bundle')
    target = Path(output_dir).resolve()
    if target.exists():
        raise FileExistsError(f'analysis output already exists: {target}')
    target.mkdir(parents=True)
    (target / 'similarities').mkdir()
    (target / 'plots').mkdir()
    metrics, table_hashes = ({}, {})
    for level in LEVELS:
        ids, matrix = _load_representation_level(representation_root, level)
        if level == 'contextual':
            expected_ids = sorted((row['occurrence_id'] for row in bundle['occurrences']))
        else:
            expected_groups = sorted((row for row in bundle['groups'] if row['group_kind'] == level), key=lambda row: row['group_id'])
            expected_ids = [row['group_id'] for row in expected_groups]
            counts = json.loads((representation_root / f'{level}_counts.json').read_text(encoding='utf-8'))
            if counts != [row['occurrence_count'] for row in expected_groups]:
                raise ValueError(f'representation group counts disagree with bundle: {level}')
        if ids != expected_ids:
            raise ValueError(f'representation IDs disagree with complete bundle objects: {level}')
        rows = cosine_rows(bundle['pairs'][level], ids, matrix)
        table = target / 'similarities' / f'{level}.jsonl'
        table.write_text(''.join((json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n' for row in rows)), encoding='utf-8')
        table_hashes[level] = _sha256(table)
        metrics[level] = summarize_rows(rows)
        _plot_bins(metrics[level], target / 'plots' / f'{level}-bin-means.png', f'{run_label}: {level}')
    (target / 'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    pair_hashes = {level: bundle['manifest']['artifacts'][f'pairs/{level}.jsonl'] for level in LEVELS}
    manifest = {'schema_version': 1, 'run_label': run_label, 'bundle_manifest_sha256': bundle_manifest_sha, 'pair_plan_sha256': pair_hashes, 'representation_manifest_sha256': _sha256(representation_root / 'manifest.json'), 'similarity_table_sha256': table_hashes, 'metrics_sha256': _sha256(target / 'metrics.json')}
    (target / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    return target

def compare_paired_runs(left_dir: str | Path, right_dir: str | Path, *, output_dir: str | Path) -> Path:
    left_root, right_root = (Path(left_dir).resolve(), Path(right_dir).resolve())
    manifests = [json.loads((root / 'manifest.json').read_text(encoding='utf-8')) for root in (left_root, right_root)]
    for root, manifest in zip((left_root, right_root), manifests):
        if _sha256(root / 'metrics.json') != manifest.get('metrics_sha256'):
            raise ValueError(f'analysis metrics checksum mismatch: {root}')
        for level in LEVELS:
            if _sha256(root / 'similarities' / f'{level}.jsonl') != manifest.get('similarity_table_sha256', {}).get(level):
                raise ValueError(f'analysis similarity checksum mismatch: {root} {level}')
    for field in ('bundle_manifest_sha256', 'pair_plan_sha256'):
        if manifests[0].get(field) != manifests[1].get(field):
            raise ValueError(f'paired runs disagree on {field}')
    metrics = [json.loads((root / 'metrics.json').read_text(encoding='utf-8')) for root in (left_root, right_root)]
    paired_tables = {}
    model_fields = {'cosine_similarity', 'hidden_dissimilarity'}
    for level in LEVELS:
        tables = []
        for root in (left_root, right_root):
            tables.append([json.loads(line) for line in (root / 'similarities' / f'{level}.jsonl').read_text(encoding='utf-8').splitlines()])
        if [row['pair_id'] for row in tables[0]] != [row['pair_id'] for row in tables[1]]:
            raise ValueError(f'paired runs have different ordered pair IDs for {level}')
        if any(({key: value for key, value in left.items() if key not in model_fields} != {key: value for key, value in right.items() if key not in model_fields} for left, right in zip(*tables))):
            raise ValueError(f'paired runs disagree on common pair metadata for {level}')
        paired_tables[level] = tables
    target = Path(output_dir).resolve()
    if target.exists():
        raise FileExistsError(f'comparison output already exists: {target}')
    target.mkdir(parents=True)
    (target / 'joined').mkdir()
    result = {'left_run': manifests[0]['run_label'], 'right_run': manifests[1]['run_label'], 'levels': {}}
    joined_hashes = {}
    for level in LEVELS:
        tables = paired_tables[level]
        joined = [{**{key: value for key, value in left.items() if key not in model_fields}, 'left_cosine_similarity': left['cosine_similarity'], 'right_cosine_similarity': right['cosine_similarity'], 'left_hidden_dissimilarity': left['hidden_dissimilarity'], 'right_hidden_dissimilarity': right['hidden_dissimilarity']} for left, right in zip(*tables)]
        joined_path = target / 'joined' / f'{level}.jsonl'
        joined_path.write_text(''.join((json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n' for row in joined)), encoding='utf-8')
        joined_hashes[level] = _sha256(joined_path)
        left_rho, right_rho = (metrics[0][level]['overall']['rho'], metrics[1][level]['overall']['rho'])
        result['levels'][level] = {'left_rho': left_rho, 'right_rho': right_rho, 'delta_rho_right_minus_left': None if left_rho is None or right_rho is None else right_rho - left_rho, 'pair_count': len(tables[0]), 'strata': {}}
        for stratum in [f'{value:03b}' for value in range(8)] + ['no_zero_initial']:
            left_value = metrics[0][level]['strata'][stratum]['rho']
            right_value = metrics[1][level]['strata'][stratum]['rho']
            result['levels'][level]['strata'][stratum] = {'left_rho': left_value, 'right_rho': right_value, 'delta_rho_right_minus_left': None if left_value is None or right_value is None else right_value - left_value}
    result['uncertainty'] = {'status': 'not_estimated', 'paired_point_difference_only': True, 'iid_pair_bootstrap': False}
    output = target / 'comparison.json'
    output.write_text(json.dumps(result, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    comparison_manifest = {'schema_version': 1, 'input_analysis_manifest_sha256': {'left': _sha256(left_root / 'manifest.json'), 'right': _sha256(right_root / 'manifest.json')}, 'comparison_sha256': _sha256(output), 'joined_sha256': joined_hashes}
    (target / 'manifest.json').write_text(json.dumps(comparison_manifest, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    return target


## Embedded `evaluation/runner.py`


In [ ]:
"""End-to-end orchestration for one immutable geometry-evaluation run."""
import hashlib
import json
import platform
import shutil
from importlib import metadata
from pathlib import Path
from typing import Any
import numpy as np

def _sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def _version(name: str) -> str | None:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

def run_evaluation(bundle_path: str | Path, *, output_root: str | Path, run_id: str, adapter: str, code_provenance: dict[str, str], experiment_path: str | Path | None=None, checkpoint_path: str | Path | None=None, device: str='cpu', batch_size: int=32, random_seed: int=42, random_dimension: int | None=None) -> Path:
    """Extract or generate representations, then run all three analyses."""
    if Path(run_id).name != run_id or run_id in {'', '.', '..'}:
        raise ValueError('run_id must be one safe path component')
    if adapter not in {'bilstm', 'random'}:
        raise ValueError("adapter must be 'bilstm' or 'random'")
    if not isinstance(code_provenance, dict) or not code_provenance or any((not isinstance(key, str) or not isinstance(value, str) or len(value) != 64 or any((character not in '0123456789abcdef' for character in value)) for key, value in code_provenance.items())):
        raise ValueError('explicit non-empty code_provenance hashes are required')
    bundle_root = Path(bundle_path).resolve()
    bundle = load_evaluation_bundle(bundle_root)
    target = (Path(output_root).resolve() / run_id).resolve()
    if target.parent != Path(output_root).resolve() or target.exists():
        raise FileExistsError(f'unsafe or existing evaluation run: {target}')
    if target == bundle_root or bundle_root in target.parents:
        raise ValueError('evaluation run output must not be inside its input bundle')
    if experiment_path is not None:
        experiment_root = Path(experiment_path).resolve()
        if target == experiment_root or experiment_root in target.parents:
            raise ValueError('evaluation run output must not be inside its source experiment')
    target.mkdir(parents=True)
    try:
        representations = target / 'representations'
        if adapter == 'bilstm':
            if experiment_path is None or checkpoint_path is None:
                raise ValueError('bilstm adapter requires experiment_path and checkpoint_path')
            extract_bilstm(bundle_root, experiment_path=experiment_path, checkpoint_path=checkpoint_path, output_dir=representations, device=device, batch_size=batch_size)
            adapter_config = {'experiment_path': str(Path(experiment_path).resolve()), 'checkpoint_path': str(Path(checkpoint_path).resolve()), 'device': device, 'batch_size': batch_size}
        else:
            if random_dimension is None or random_dimension <= 0 or isinstance(random_dimension, bool):
                raise ValueError('random adapter requires a positive random_dimension')
            occurrence_ids = sorted((row['occurrence_id'] for row in bundle['occurrences']))
            hidden = random_occurrence_baseline(occurrence_ids, dimension=random_dimension, seed=random_seed)
            save_representations(representations, occurrence_ids=occurrence_ids, hidden=hidden, groups=bundle['groups'], provenance={'adapter': 'iid_standard_normal', 'seed': random_seed, 'dimension': random_dimension, 'bundle_manifest_sha256': _sha256(bundle_root / 'manifest.json')})
            adapter_config = {'random_seed': random_seed, 'random_dimension': random_dimension}
        analysis = analyze_representations(bundle_root, representations, output_dir=target / 'analysis', run_label=run_id)
        actual = {'schema_version': 1, 'run_id': run_id, 'adapter': adapter, 'adapter_config': adapter_config, 'bundle_path': str(bundle_root), 'bundle_manifest_sha256': _sha256(bundle_root / 'manifest.json'), 'representations_manifest_sha256': _sha256(representations / 'manifest.json'), 'analysis_manifest_sha256': _sha256(analysis / 'manifest.json'), 'code_provenance': code_provenance, 'environment': {'python': platform.python_version(), 'numpy': np.__version__, 'torch': _version('torch'), 'dimsim': _version('dimsim'), 'pypinyin': _version('pypinyin')}}
        (target / 'actual_config.json').write_text(json.dumps(actual, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    except Exception:
        shutil.rmtree(target)
        raise
    return target


In [ ]:
from pathlib import Path

EXPERIMENT_ID = "1.0.0"
SEED = 42
LAMBDA_MSE = 1.0

# Kaggle examples; edit these explicit paths for attached datasets.
BUNDLE_PATH = Path("/kaggle/input/mandarin-evaluation-1-0-0/1.0.0")
OUTPUT_ROOT = Path("/kaggle/working/evaluation-runs")
EXPERIMENT_PATH = Path("/kaggle/input/mandarin-experiment-1-0-0") / EXPERIMENT_ID
RUNS_ROOT = Path("/kaggle/input/mandarin-training-runs/runs") / EXPERIMENT_ID

RUNS = [
    {"run_id": "lambda-0", "adapter": "bilstm", "experiment_path": EXPERIMENT_PATH, "checkpoint_path": RUNS_ROOT / "ce_only" / f"seed-{SEED}" / "best.pt"},
    {"run_id": "lambda-1.0", "adapter": "bilstm", "experiment_path": EXPERIMENT_PATH, "checkpoint_path": RUNS_ROOT / "combined" / f"lambda-{LAMBDA_MSE:g}" / f"seed-{SEED}" / "best.pt"},
    {"run_id": "random-seed-42", "adapter": "random", "random_seed": 42, "random_dimension": 1024},
]
DEVICE = "cpu"
BATCH_SIZE = 32
PAIRED_COMPARISONS = [("lambda-0", "lambda-1.0")]


In [ ]:
import torch
print("torch", torch.__version__, "cuda_available", torch.cuda.is_available())
if DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError(f"Selected device {DEVICE!r} requires CUDA, but CUDA is unavailable")


In [ ]:
missing_checkpoints = [str(specification["checkpoint_path"]) for specification in RUNS
                       if specification["adapter"] != "random" and not Path(specification["checkpoint_path"]).is_file()]
if missing_checkpoints:
    raise FileNotFoundError("Training checkpoints are required before evaluation: " + ", ".join(missing_checkpoints))

completed = {}
for specification in RUNS:
    arguments = dict(specification)
    run_id = arguments.pop("run_id")
    adapter = arguments.pop("adapter")
    completed[run_id] = run_evaluation(
        BUNDLE_PATH, output_root=OUTPUT_ROOT, run_id=run_id, adapter=adapter,
        device=DEVICE, batch_size=BATCH_SIZE, code_provenance=BUILD_METADATA["source_sha256"], **arguments,
    )

for left, right in PAIRED_COMPARISONS:
    compare_paired_runs(
        completed[left] / "analysis", completed[right] / "analysis",
        output_dir=OUTPUT_ROOT / f"compare-{left}-vs-{right}",
    )

for run_id, path in completed.items():
    metrics = json.loads((path / "analysis" / "metrics.json").read_text(encoding="utf-8"))
    print(run_id, {level: values["overall"] for level, values in metrics.items()})
